In [1]:
!pip install pyfaidx transformers datasets tqdm
!pip install transformers accelerate --quiet

import pandas as pd
import requests
from tqdm import tqdm
from pyfaidx import Fasta
from transformers import BertTokenizer, BertForSequenceClassification
import torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 15.7 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system =

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [2]:
# ========================
#  📌 第二步：读取 VCF 文件（你需要在 Colab 手动上传）
# ========================
vcf_filename = "ClinVar_Coding_SNV_PB.vcf"  # 你需要替换为自己的 VCF 文件名

''' # 解析 VCF 文件
vcf_data = []
with open(vcf_filename, "r") as file:
    for line in file:
        if not line.startswith("#"):  # 跳过注释行
            fields = line.strip().split("\t")
            chrom, pos, ref, alt = fields[0], int(fields[1]), fields[3], fields[4]
            vcf_data.append([chrom, pos, ref, alt]) '''

# 解析 VCF 文件，同时提取 `INFO` 字段
vcf_data = []
with open(vcf_filename, "r") as file:
    for line in file:
        if not line.startswith("#"):  # 跳过注释行
            fields = line.strip().split("\t")

            # 提取关键字段
            chrom, pos, ref, alt, info = fields[0], int(fields[1]), fields[3], fields[4], fields[7]

            # 将数据存入列表
            vcf_data.append([chrom, pos, ref, alt, info])

# 创建 DataFrame
df_vcf = pd.DataFrame(vcf_data, columns=["CHROM", "POS", "REF", "ALT", "INFO"])


#df_vcf = pd.DataFrame(vcf_data, columns=["CHROM", "POS", "REF", "ALT"])

# 生成 True_Label（1=致病, 0=良性）
#df_vcf["True_Label"] = df_vcf["INFO"]

print("✅ 解析 VCF 完成！")




✅ 解析 VCF 完成！


In [3]:
df_vcf

,CHROM,POS,REF,ALT,INFO
0,11,126275389,C,T,1.0
1,11,126277517,A,G,1.0
2,6,26093215,G,T,1.0
3,2,19945787,T,C,1.0
4,20,25302322,G,A,1.0
...,...,...,...,...,...
136920,5,75416973,G,A,1.0
136921,X,41346238,C,G,1.0
136922,7,140801551,T,C,1.0
136923,9,121314019,A,G,1.0


In [4]:
# ========================
#  📌 第三步：下载 GRCh38 参考基因组
# ========================
!wget -c http://hgdownload.cse.ucsc.edu/goldenpath/hg38/bigZips/hg38.fa.gz
!gunzip -k hg38.fa.gz

--2025-04-02 13:29:36--  http://hgdownload.cse.ucsc.edu/goldenpath/hg38/bigZips/hg38.fa.gz
Resolving hgdownload.cse.ucsc.edu (hgdownload.cse.ucsc.edu)... 128.114.119.163
Connecting to hgdownload.cse.ucsc.edu (hgdownload.cse.ucsc.edu)|128.114.119.163|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 983659424 (938M) [application/x-gzip]
Saving to: ‘hg38.fa.gz’

hg38.fa.gz          100%[===================>] 938.09M  19.9MB/s    in 49s     

2025-04-02 13:30:25 (19.2 MB/s) - ‘hg38.fa.gz’ saved [983659424/983659424]



In [5]:
# 加载参考基因组
genome = Fasta("hg38.fa")

# ========================
#  📌 第四步：提取突变上下游 50bp 序列（共 101bp）
# ========================
window = 128

def get_sequence(chrom, pos, ref, alt, flank_size=window):
    """ 获取突变上下游 50bp 的基因组序列（共 101bp） """
    chrom = "chr" + chrom
    start = max(0, pos - flank_size - 1)  # UCSC 坐标是 0-based
    end = pos + flank_size
    seq = genome[chrom][start:end].seq.upper()  # 提取上下游序列
    return seq

# 提取所有突变位点的序列
sequences = []
for _, row in tqdm(df_vcf.iterrows(), total=len(df_vcf)):
    chrom, pos, ref, alt = row["CHROM"], row["POS"], row["REF"], row["ALT"]
    seq = get_sequence(chrom, pos, ref, alt)
    sequences.append(seq)

df_vcf["Context_Sequence"] = sequences

def generate_mutant_sequence(row):
    """
    用 ALT 替换 Context_Sequence 中 REF 的位置，生成突变序列
    """
    seq = list(row["Context_Sequence"])  # 转换为列表以进行修改
    mut_pos = window  # 突变发生在中心位置（上下游各 50bp）

    # 进行突变替换（仅支持单碱基突变）
    if seq[mut_pos] == row["REF"]:
        seq[mut_pos] = row["ALT"]
    return "".join(seq)

# 生成突变序列
df_vcf["Mutant_Sequence"] = df_vcf.apply(generate_mutant_sequence, axis=1)

#df_vcf.to_csv("processed_vcf_with_sequences.csv", index=False)
print("✅ 突变序列提取完成！")


# ========================
#  📌 第五步：转换为 DNABERT2 的 k-mer 格式
# ========================
from tqdm import tqdm

# 启用 tqdm 进度条
tqdm.pandas()

def generate_kmers(sequence, k=6):
    """ 将序列转换为 k-mer 格式（适用于 DNABERT2） """
    return " ".join([sequence[i:i+k] for i in range(len(sequence) - k + 1)])

# 添加进度条
print("🔄 正在转换 Context_Sequence 为 k-mer 格式...")
df_vcf["Kmer_Sequence"] = df_vcf["Context_Sequence"].progress_apply(lambda x: generate_kmers(x, k=6))

print("🔄 正在转换 Mutant_Sequence 为 k-mer 格式...")
df_vcf["Kmer_Sequence_Mutant"] = df_vcf["Mutant_Sequence"].progress_apply(lambda x: generate_kmers(x, k=6))

# 保存结果
df_vcf.to_csv("dnabert2_input.csv", index=False)
print("✅ k-mer 转换完成！")




100%|██████████| 136925/136925 [00:08<00:00, 15419.45it/s]


✅ 突变序列提取完成！
🔄 正在转换 Context_Sequence 为 k-mer 格式...


100%|██████████| 136925/136925 [00:03<00:00, 34936.41it/s]


🔄 正在转换 Mutant_Sequence 为 k-mer 格式...


100%|██████████| 136925/136925 [00:03<00:00, 34563.10it/s]


✅ k-mer 转换完成！


In [6]:
df = pd.read_csv("processed_vcf_with_sequences.csv")
df = df[['CHROM', 'POS', 'REF', 'ALT', 'INFO', 'Context_Sequence', 'Mutant_Sequence']]
df

FileNotFoundError: [Errno 2] No such file or directory: 'processed_vcf_with_sequences.csv'

In [7]:
df_vcf

,CHROM,POS,REF,ALT,INFO,Context_Sequence,Mutant_Sequence,Kmer_Sequence,Kmer_Sequence_Mutant
0,11,126275389,C,T,1.0,CAGGAAATACAATCCAAGAGCAGAAGTCCTCATCCCTCTTTGTGAG...,CAGGAAATACAATCCAAGAGCAGAAGTCCTCATCCCTCTTTGTGAG...,CAGGAA AGGAAA GGAAAT GAAATA AAATAC AATACA ATAC...,CAGGAA AGGAAA GGAAAT GAAATA AAATAC AATACA ATAC...
1,11,126277517,A,G,1.0,GTGGCTACAGCCTTCCCGAGAACCCCAGTGTTTTGTGCACCCGCAG...,GTGGCTACAGCCTTCCCGAGAACCCCAGTGTTTTGTGCACCCGCAG...,GTGGCT TGGCTA GGCTAC GCTACA CTACAG TACAGC ACAG...,GTGGCT TGGCTA GGCTAC GCTACA CTACAG TACAGC ACAG...
2,6,26093215,G,T,1.0,TCAAAGGCTTTAACTTGCTTTTTCTGTTTTAGAGCCCTCACCGTCT...,TCAAAGGCTTTAACTTGCTTTTTCTGTTTTAGAGCCCTCACCGTCT...,TCAAAG CAAAGG AAAGGC AAGGCT AGGCTT GGCTTT GCTT...,TCAAAG CAAAGG AAAGGC AAGGCT AGGCTT GGCTTT GCTT...
3,2,19945787,T,C,1.0,AGAAATTCATGCAGCCGTGTTGGCACTGCTATTCAACAAACTTCTT...,AGAAATTCATGCAGCCGTGTTGGCACTGCTATTCAACAAACTTCTT...,AGAAAT GAAATT AAATTC AATTCA ATTCAT TTCATG TCAT...,AGAAAT GAAATT AAATTC AATTCA ATTCAT TTCATG TCAT...
4,20,25302322,G,A,1.0,CCTGGGTGGGAAGAGAATGTCTCACCTCAGTATCCGTGGCAGCTCA...,CCTGGGTGGGAAGAGAATGTCTCACCTCAGTATCCGTGGCAGCTCA...,CCTGGG CTGGGT TGGGTG GGGTGG GGTGGG GTGGGA TGGG...,CCTGGG CTGGGT TGGGTG GGGTGG GGTGGG GTGGGA TGGG...
...,...,...,...,...,...,...,...,...,...
136920,5,75416973,G,A,1.0,TATTTTTAAAAGGAAAAAAACCTAGTCTTACCTTATCCAGTCTCTT...,TATTTTTAAAAGGAAAAAAACCTAGTCTTACCTTATCCAGTCTCTT...,TATTTT ATTTTT TTTTTA TTTTAA TTTAAA TTAAAA TAAA...,TATTTT ATTTTT TTTTTA TTTTAA TTTAAA TTAAAA TAAA...
136921,X,41346238,C,G,1.0,AAGCCCGTTTTTAAGAAGATATATATGTATTTTAATTGACACATTA...,AAGCCCGTTTTTAAGAAGATATATATGTATTTTAATTGACACATTA...,AAGCCC AGCCCG GCCCGT CCCGTT CCGTTT CGTTTT GTTT...,AAGCCC AGCCCG GCCCGT CCCGTT CCGTTT CGTTTT GTTT...
136922,7,140801551,T,C,1.0,ATAATTAACACACATCAGTGGAACTTCTGTACTACAACGCTGGTGA...,ATAATTAACACACATCAGTGGAACTTCTGTACTACAACGCTGGTGA...,ATAATT TAATTA AATTAA ATTAAC TTAACA TAACAC AACA...,ATAATT TAATTA AATTAA ATTAAC TTAACA TAACAC AACA...
136923,9,121314019,A,G,1.0,CCCTCACAGCCACCCTTCCTCTCCATCTCTCTATCTCCTACAGGTG...,CCCTCACAGCCACCCTTCCTCTCCATCTCTCTATCTCCTACAGGTG...,CCCTCA CCTCAC CTCACA TCACAG CACAGC ACAGCC CAGC...,CCCTCA CCTCAC CTCACA TCACAG CACAGC ACAGCC CAGC...


In [ ]:
df_1 = df_vcf

In [ ]:
df_vcf = df_1

In [11]:
df_vcf = df_vcf.head(5000)

In [12]:
# ========================
#  📌 第六步：使用 DNABERT2 进行预测
# =======================


from transformers import BertModel, AutoTokenizer

model = BertModel.from_pretrained("zhihan1996/DNABERT-2-117M", trust_remote_code=True)

# ✅ 解决 DNABERT2 的 `config_class` 兼容性问题
tokenizer = AutoTokenizer.from_pretrained("zhihan1996/DNABERT-2-117M", trust_remote_code=True)

from tqdm.notebook import tqdm  # ✅ 替代原始 tqdm


def predict_sequence(sequence):
    """ 使用 DNABERT2 计算句子的向量表示 """
    inputs = tokenizer(sequence, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state[:, 0, :].tolist()[0]  # 取 [CLS] token 的向量表示

# 使用 tqdm 显示进度条
predictions = []
for _, row in tqdm(df_vcf.iterrows(), total=len(df_vcf), leave=True, dynamic_ncols=True,desc="Predicting with DNABERT2"):
    pred = predict_sequence(row["Kmer_Sequence"])
    predictions.append(pred)

# 添加预测结果
df_vcf["DNABERT2_Predictions"] = predictions

predictions_m = []
for _, row in tqdm(df_vcf.iterrows(), total=len(df_vcf), leave=True, dynamic_ncols=True, desc="Predicting with DNABERT2 (Mutatnt)"):
    pred_m = predict_sequence(row["Kmer_Sequence_Mutant"])
    predictions_m.append(pred_m)

df_vcf["DNABERT2_Predictions_Mutant"] = predictions_m
# 保存结果
df_vcf.to_csv("dnabert2_predictions.csv", index=False)
print("✅ DNABERT2 预测完成！结果已保存：dnabert2_predictions.csv")


Some weights of BertModel were not initialized from the model checkpoint at zhihan1996/DNABERT-2-117M and are newly initialized: ['embeddings.position_embeddings.weight', 'encoder.layer.0.attention.self.key.bias', 'encoder.layer.0.attention.self.key.weight', 'encoder.layer.0.attention.self.query.bias', 'encoder.layer.0.attention.self.query.weight', 'encoder.layer.0.attention.self.value.bias', 'encoder.layer.0.attention.self.value.weight', 'encoder.layer.0.intermediate.dense.bias', 'encoder.layer.0.intermediate.dense.weight', 'encoder.layer.0.output.LayerNorm.bias', 'encoder.layer.0.output.LayerNorm.weight', 'encoder.layer.0.output.dense.bias', 'encoder.layer.0.output.dense.weight', 'encoder.layer.1.attention.self.key.bias', 'encoder.layer.1.attention.self.key.weight', 'encoder.layer.1.attention.self.query.bias', 'encoder.layer.1.attention.self.query.weight', 'encoder.layer.1.attention.self.value.bias', 'encoder.layer.1.attention.self.value.weight', 'encoder.layer.1.intermediate.dense.b

Predicting with DNABERT2:   0%|          | 0/5000 [00:00<?, ?it/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Predicting with DNABERT2 (Mutatnt):   0%|          | 0/5000 [00:00<?, ?it/s]

✅ DNABERT2 预测完成！结果已保存：dnabert2_predictions.csv


In [15]:
import ast
# 重新导入必要的库
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score

# 重新加载文件
file_path = "dnabert2_predictions.csv"

# 读取 DNABERT2 预测的结果文件
df_predictions = pd.read_csv(file_path)


def compute_llr(prediction_list):
    values = np.array(ast.literal_eval(prediction_list))  # 解析字符串列表
    return np.sum(values)  # 计算总和


# 计算参考序列的 log-likelihood
df_predictions["LLR_Ref"] = df_predictions["DNABERT2_Predictions"].apply(compute_llr)
df_predictions["LLR_Mut"] = df_predictions["DNABERT2_Predictions_Mutant"].apply(compute_llr)

# 计算 LLR 差值
df_predictions["Delta_LLR"] = df_predictions["LLR_Mut"] - df_predictions["LLR_Ref"]

auc_score = roc_auc_score(df_predictions["INFO"], df_predictions["Delta_LLR"])
print(f"Zero-shot 预测 AUC: {auc_score:.4f}")

Zero-shot 预测 AUC: 0.5143


In [16]:
# ========================
# ✅ 加载 NT v2 模型
# ========================
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM

from tqdm.notebook import tqdm  # ✅ 替代原始 tqdm

model_name = "InstaDeepAI/nucleotide-transformer-v2-50m-multi-species"

# 加载 tokenizer 和 模型，信任远程自定义代码
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForMaskedLM.from_pretrained(model_name, trust_remote_code=True)
model.eval()

# ========================
# ✅ 提取 embedding 的函数（平均池化）
# ========================
def predict_sequence(sequence):
    inputs = tokenizer(sequence, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
        embedding = outputs.hidden_states[-1].mean(dim=1).squeeze().tolist()
    return embedding

# ========================
# ✅ Wild-type embedding 提取
# ========================
predictions = []
for _, row in tqdm(df_vcf.iterrows(), total=len(df_vcf), desc="Predicting with NT v2"):
    pred = predict_sequence(row["Kmer_Sequence"])
    predictions.append(pred)

df_vcf["DNABERT2_Predictions"] = predictions  # 不改列名，复用原结构

# ========================
# ✅ Mutant embedding 提取
# ========================
predictions_m = []
for _, row in tqdm(df_vcf.iterrows(), total=len(df_vcf), desc="Predicting with NT v2 (Mutant)"):
    pred_m = predict_sequence(row["Kmer_Sequence_Mutant"])
    predictions_m.append(pred_m)

df_vcf["DNABERT2_Predictions_Mutant"] = predictions_m

# ========================
# ✅ 保存结果
# ========================
#df_vcf.to_csv("dnabert2_predictions.csv", index=False)
print("✅ NT v2 预测完成！结果已保存：dnabert2_predictions.csv")


tokenizer_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/28.7k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/101 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

esm_config.py:   0%|          | 0.00/14.9k [00:00<?, ?B/s]

...

A new version of the following files was downloaded from https://huggingface.co/InstaDeepAI/nucleotide-transformer-v2-50m-multi-species:
- modeling_esm.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/224M [00:00<?, ?B/s]

Predicting with NT v2:   0%|          | 0/5000 [00:00<?, ?it/s]

Predicting with NT v2 (Mutant):   0%|          | 0/5000 [00:00<?, ?it/s]

✅ NT v2 预测完成！结果已保存：dnabert2_predictions.csv


In [17]:
import numpy as np
from sklearn.metrics import roc_auc_score
from sklearn.metrics.pairwise import cosine_similarity

# 将 embedding 转换为矩阵
ref_embeddings = np.array(df_vcf["DNABERT2_Predictions"].to_list())
mut_embeddings = np.array(df_vcf["DNABERT2_Predictions_Mutant"].to_list())

# 计算余弦相似度（越小越可能致病）
cos_sim = np.array([
    cosine_similarity(ref.reshape(1, -1), mut.reshape(1, -1))[0, 0]
    for ref, mut in zip(ref_embeddings, mut_embeddings)
])

# 使用 1 - cos_sim 作为 deleterious 分数（越大越可能致病）
deleterious_scores = 1 - cos_sim

# 获取真实标签
true_labels = df_vcf["INFO"].values

# 计算 AUC
auc_score = roc_auc_score(true_labels, deleterious_scores)
print(auc_score)


0.5893275423178737
